# Manuelle Manipulation von Spektren

Dieses Notebook lädt die Originaldaten, interpoliert sie wie im Interpolationsnotebook und erlaubt manuelle Entfernen-Regeln für ganze Zeitbereiche oder einzelne Frequenzbereiche. Die Originaldaten werden nicht verändert; manipulierte Tabellen können separat exportiert und später wieder geladen werden.

## Einstellungen und manuelle Regeln

In [ ]:
from pathlib import Path
import importlib
import json
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 6),
    "axes.grid": True,
})

COLUMNS = [
    "Freq_Hz",
    "Time_s",
    "Eps_real",
    "Eps_imag",
    "Temp_K",
    "MTime_s",
    "Phi_deg",
    "Z_real_Ohm",
    "Z_imag_Ohm",
    "TanPhi",
]

TIME_COL = "Time_Relative_s"
INTERPOLATED_VALUE_COLS = ("Eps_real",)

MATERIAL = "PEI5mgmL"
TEMPERATURE = "70°C"
MODE = "Des"
PLOT_TIME_S = 4000

# Fuer diese manuelle Manipulation wird die 50C-Abs-Serie direkt geladen.
SELECTED_FILE = f"{MATERIAL}_{TEMPERATURE}_{MODE}.TXT"
MATERIAL_FILTER = MATERIAL

# Export der final manipulierten Ableitungsdaten wird in der letzten Zelle bewusst aktiviert.
MANIPULATION_NAME = "manual_v1"
EXPORT_MANIPULATED_DATA = False

# Ableitungs-Regeln entfernen Punkte erst nach der Ableitungsbildung aus df_derivative_original.
# Unterstützte Felder wie oben, aber Frequenzspalten beziehen sich auf Freq_Hz_mid.
# Zusätzlich: omega_min_rad_s, omega_max_rad_s, derivative_point_index.
MANUAL_DERIVATIVE_REMOVALS = [
    {
        "enabled": True,
        "reason": "Zweiter Ableitungspunkt von hinten 70C Des manuell entfernt",
        "material": "PEI5mgmL",
        "temperature": "70°C",
        "mode": "Des",
        "derivative_point_from_end": -2,
    },
    {
        "enabled": True,
        "reason": "Dritter Ableitungspunkt von hinten 70C Des manuell entfernt",
        "material": "PEI5mgmL",
        "temperature": "70°C",
        "mode": "Des",
        "derivative_point_from_end": -3,
    },
    {
        "enabled": True,
        "reason": "Zweiter Ableitungspunkt von vorne 70C Des manuell entfernt",
        "material": "PEI5mgmL",
        "temperature": "70°C",
        "mode": "Des",
        "derivative_point_index": 1,
    },
    # Beispiel: Ableitungspunkte am Rand eines Offset-Sprungs entfernen.
    # {
    #     "enabled": False,
    #     "reason": "Ableitungs-Spike durch Offset-Sprung",
    #     "material": "PEI5mgmL",
    #     "temperature": "80°C",
    #     "mode": "Abs",
    #     "time_min_s": 0,
    #     "time_max_s": 3000,
    #     "freq_min_hz": 4.0e5,
    #     "freq_max_hz": 8.0e5,
    # },
]


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data" / "Daten final").exists():
            return path
    raise FileNotFoundError("Could not find the project root with data/Daten final.")


def add_code_dir_to_path(project_root):
    code_dir = project_root / "Code"
    code_dir_text = str(code_dir)
    if code_dir_text not in sys.path:
        sys.path.insert(0, code_dir_text)


PROJECT_ROOT = find_project_root()
add_code_dir_to_path(PROJECT_ROOT)
import switch_points
importlib.reload(switch_points)
manual_switch_points = switch_points.MANUAL_SWITCH_POINTS

DATA_DIR = PROJECT_ROOT / "data" / "Daten final"
EXPORT_DIR = PROJECT_ROOT / "data" / "data_manipulated"
DATA_DIR

## Daten laden und interpolieren

In [ ]:
def load_measurements(data_dir=DATA_DIR, selected_file=None, material_filter="PEI5mgmL"):
    if selected_file is None:
        file_paths = sorted(data_dir.glob("*.TXT")) + sorted(data_dir.glob("*.txt"))
    else:
        file_path = data_dir / selected_file
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        file_paths = [file_path]

    if selected_file is None and material_filter is not None:
        file_paths = [path for path in file_paths if path.stem.split("_")[0] == material_filter]

    if not file_paths:
        raise FileNotFoundError(f"No .TXT files found in: {data_dir}")

    frames = []
    for path in file_paths:
        df = pd.read_csv(
            path,
            sep=r"\s+",
            skiprows=4,
            names=COLUMNS,
            encoding="latin1",
            engine="python",
        )
        parts = path.stem.split("_")
        material, temperature, mode = parts[:3] if len(parts) >= 3 else (path.stem, None, None)
        first_freq = df["Freq_Hz"].iloc[0]
        df["Spectrum_ID"] = (df["Freq_Hz"] == first_freq).cumsum() - 1
        df["Spectrum_Number"] = df["Spectrum_ID"] + 1
        df["Point_Number"] = range(1, len(df) + 1)
        df["Material"] = material
        df["Temperature"] = temperature
        df["Mode"] = mode
        df["Source_File"] = path.name
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def add_relative_switch_time(df, switch_points):
    df = df.copy()
    switch_time_rows = []
    for dataset_key, dataset in df.groupby(["Material", "Temperature", "Mode"], dropna=False):
        switch_point_number = switch_points.get(dataset_key)
        if switch_point_number is None:
            raise ValueError(f"Missing Switch_Point_Number for series: {dataset_key}")
        switch_point = dataset[dataset["Point_Number"] == switch_point_number]
        next_point = dataset[dataset["Point_Number"] == switch_point_number + 1]
        if switch_point.empty or next_point.empty:
            raise ValueError(f"Switch-point pair not found: {dataset_key}, {switch_point_number}")
        switch_time_rows.append({
            "Material": dataset_key[0],
            "Temperature": dataset_key[1],
            "Mode": dataset_key[2],
            "Switch_Point_Number": int(switch_point_number),
            "Switch_Spectrum_ID": int(switch_point["Spectrum_ID"].iloc[0]),
            "Switch_Time_s": float((switch_point["MTime_s"].iloc[0] + next_point["MTime_s"].iloc[0]) / 2),
        })
    switch_times = pd.DataFrame(switch_time_rows)
    df = df.merge(switch_times, on=["Material", "Temperature", "Mode"], how="left")
    df[TIME_COL] = df["MTime_s"] - df["Switch_Time_s"]
    return df, switch_times


def _linear_at(target_time, times, values):
    if len(times) < 2:
        return np.nan
    x0, x1 = float(times[0]), float(times[1])
    y0, y1 = float(values[0]), float(values[1])
    if x0 == x1:
        return np.nan
    return y0 + (target_time - x0) * (y1 - y0) / (x1 - x0)


def _insert_switch_support_point(freq_data, value_col, time_col=TIME_COL, target_time=0):
    before = freq_data[freq_data[time_col] < target_time].tail(2)
    support_points = freq_data[[time_col, value_col]].dropna().copy()
    support_points = support_points[~np.isclose(support_points[time_col], target_time)]
    if len(before) >= 2:
        switch_value = _linear_at(target_time, before[time_col].to_numpy(), before[value_col].to_numpy())
        support_points = pd.concat(
            [support_points, pd.DataFrame({time_col: [float(target_time)], value_col: [switch_value]})],
            ignore_index=True,
        )
    support_points = support_points.sort_values(time_col)
    return support_points.groupby(time_col, as_index=False)[value_col].mean()


def interpolate_complete_spectra(
    df,
    time_col=TIME_COL,
    freq_col="Freq_Hz",
    value_cols=INTERPOLATED_VALUE_COLS,
    dataset_cols=("Source_File", "Material", "Temperature", "Mode"),
    drop_incomplete=True,
    include_switch_time=True,
):
    rows = []
    metadata_cols = ["Switch_Point_Number", "Switch_Spectrum_ID", "Switch_Time_s"]
    for dataset_key, dataset in df.groupby(list(dataset_cols), dropna=False):
        dataset_key = dataset_key if isinstance(dataset_key, tuple) else (dataset_key,)
        dataset_meta = dict(zip(dataset_cols, dataset_key))
        target_times = np.sort(dataset[time_col].dropna().unique())
        if include_switch_time and not np.isclose(target_times, 0).any():
            target_times = np.sort(np.append(target_times, 0.0))
        frequencies = np.sort(dataset[freq_col].dropna().unique())
        dataset_metadata = {
            col: dataset[col].dropna().iloc[0]
            for col in metadata_cols
            if col in dataset.columns and not dataset[col].dropna().empty
        }

        interpolated_by_freq = {}
        for freq in frequencies:
            freq_data = dataset.loc[dataset[freq_col] == freq, [time_col, *value_cols]].dropna(subset=[time_col])
            freq_data = freq_data.sort_values(time_col).groupby(time_col, as_index=False)[list(value_cols)].mean()
            interpolated_values = {}
            for value_col in value_cols:
                support = _insert_switch_support_point(freq_data, value_col, time_col=time_col, target_time=0)
                interpolated_values[value_col] = np.interp(
                    target_times,
                    support[time_col].to_numpy(),
                    support[value_col].to_numpy(),
                    left=np.nan,
                    right=np.nan,
                )
            interpolated_by_freq[freq] = interpolated_values

        for spectrum_id, target_time in enumerate(target_times):
            for freq in frequencies:
                row = {
                    **dataset_meta,
                    **dataset_metadata,
                    "Interpolated_Spectrum_ID": spectrum_id,
                    time_col: target_time,
                    freq_col: freq,
                    "Is_Switch_Spectrum": bool(np.isclose(target_time, 0)),
                }
                for value_col in value_cols:
                    row[value_col] = interpolated_by_freq[freq][value_col][spectrum_id]
                rows.append(row)

    interpolated = pd.DataFrame(rows)
    if drop_incomplete:
        complete_ids = [*dataset_cols, "Interpolated_Spectrum_ID"]
        complete_mask = interpolated.groupby(complete_ids, dropna=False)[list(value_cols)].transform(
            lambda values: values.notna().all()
        )
        interpolated = interpolated[complete_mask.all(axis=1)].reset_index(drop=True)
    return interpolated


df_raw = load_measurements(selected_file=SELECTED_FILE, material_filter=MATERIAL_FILTER)
df_raw, switch_times = add_relative_switch_time(df_raw, manual_switch_points)
df_interpolated_complete = interpolate_complete_spectra(df_raw)

print(f"Loaded rows: {len(df_raw):,}")
print(f"Interpolated rows: {len(df_interpolated_complete):,}")
print(switch_times.to_string(index=False))

## Regeln anwenden

In [ ]:
def _as_list(value):
    if value is None:
        return None
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return list(value)
    return [value]


def _match_optional_exact(series, rule, rule_key):
    values = _as_list(rule.get(rule_key))
    if values is None:
        return pd.Series(True, index=series.index)
    return series.isin(values)


def build_rule_mask(df, rule, kind="spectrum"):
    mask = pd.Series(True, index=df.index)

    column_rule_pairs = [
        ("Source_File", "source_file"),
        ("Material", "material"),
        ("Temperature", "temperature"),
        ("Mode", "mode"),
        ("Interpolated_Spectrum_ID", "spectrum_id"),
    ]
    for column, rule_key in column_rule_pairs:
        if column in df.columns:
            mask &= _match_optional_exact(df[column], rule, rule_key)

    if "time_min_s" in rule:
        mask &= df[TIME_COL] >= float(rule["time_min_s"])
    if "time_max_s" in rule:
        mask &= df[TIME_COL] <= float(rule["time_max_s"])

    freq_col = "Freq_Hz" if kind == "spectrum" else "Freq_Hz_mid"
    if "freq_hz" in rule:
        targets = np.asarray(_as_list(rule["freq_hz"]), dtype=float)
        local = pd.Series(False, index=df.index)
        for target in targets:
            local |= np.isclose(df[freq_col].astype(float), target, rtol=1e-6, atol=0)
        mask &= local
    if "freq_min_hz" in rule:
        mask &= df[freq_col] >= float(rule["freq_min_hz"])
    if "freq_max_hz" in rule:
        mask &= df[freq_col] <= float(rule["freq_max_hz"])

    if kind == "derivative":
        if "omega_min_rad_s" in rule:
            mask &= df["Omega_rad_s"] >= float(rule["omega_min_rad_s"])
        if "omega_max_rad_s" in rule:
            mask &= df["Omega_rad_s"] <= float(rule["omega_max_rad_s"])
        if "derivative_point_index" in rule:
            mask &= _match_optional_exact(df["Derivative_Point_Index"], rule, "derivative_point_index")
        if "derivative_point_from_end" in rule:
            offset = int(rule["derivative_point_from_end"])
            group_cols = ["Source_File", "Material", "Temperature", "Mode", "Interpolated_Spectrum_ID", TIME_COL]
            available_group_cols = [col for col in group_cols if col in df.columns]
            last_index = df.groupby(available_group_cols, dropna=False)["Derivative_Point_Index"].transform("max")
            target_index = last_index + 1 + offset if offset < 0 else offset
            mask &= df["Derivative_Point_Index"] == target_index

    return mask.fillna(False)


def apply_manual_removals(df, rules, kind="spectrum", flag_col="Manual_Removed", reason_col="Manual_Removal_Reason"):
    marked = df.copy()
    marked[flag_col] = False
    marked[reason_col] = ""

    for i, rule in enumerate(rules):
        if not rule.get("enabled", True):
            continue
        mask = build_rule_mask(marked, rule, kind=kind)
        reason = rule.get("reason", f"manual_rule_{i}")
        marked.loc[mask, flag_col] = True
        old_reason = marked.loc[mask, reason_col].astype(str)
        marked.loc[mask, reason_col] = np.where(old_reason == "", reason, old_reason + "; " + reason)

    return marked


# Die Spektren selbst bleiben unverändert. Manuell entfernt wird erst in der Ableitung.
df_interpolated_reference = df_interpolated_complete.copy()

print(f"Interpolated reference points: {len(df_interpolated_reference):,}")

## Ableitung berechnen und erst danach manuell manipulieren

In [ ]:
def derive_eps_real_by_frequency(
    df,
    value_col="Eps_real",
    freq_col="Freq_Hz",
    time_col=TIME_COL,
    spectrum_col="Interpolated_Spectrum_ID",
    dataset_cols=("Source_File", "Material", "Temperature", "Mode"),
):
    rows = []
    group_cols = [*dataset_cols, spectrum_col, time_col]

    for group_key, spectrum in df.groupby(group_cols, dropna=False):
        spectrum = spectrum.sort_values(freq_col).reset_index(drop=True)
        if len(spectrum) < 2:
            continue

        frequencies = spectrum[freq_col].to_numpy()
        eps_real = spectrum[value_col].to_numpy()
        omega = 2 * np.pi * frequencies
        ln_omega = np.log(omega)
        derivative = -np.pi / 2 * np.diff(eps_real) / np.diff(ln_omega)
        omega_mid = np.exp((ln_omega[:-1] + ln_omega[1:]) / 2)
        freq_mid = omega_mid / (2 * np.pi)

        group_key = group_key if isinstance(group_key, tuple) else (group_key,)
        meta = dict(zip(group_cols, group_key))
        for i, (freq, omega_value, derivative_value) in enumerate(zip(freq_mid, omega_mid, derivative)):
            rows.append({
                **meta,
                "Derivative_Point_Index": i,
                "Freq_Hz_left": frequencies[i],
                "Freq_Hz_right": frequencies[i + 1],
                "Eps_real_left": eps_real[i],
                "Eps_real_right": eps_real[i + 1],
                "Freq_Hz_mid": freq,
                "Omega_rad_s": omega_value,
                "Eps_real_derivative": derivative_value,
            })

    return pd.DataFrame(rows)


df_derivative_original = derive_eps_real_by_frequency(df_interpolated_reference)

df_derivative_marked = apply_manual_removals(
    df_derivative_original,
    MANUAL_DERIVATIVE_REMOVALS,
    kind="derivative",
    flag_col="Derivative_Manual_Removed",
    reason_col="Derivative_Manual_Removal_Reason",
)
df_derivative_manual = df_derivative_marked[~df_derivative_marked["Derivative_Manual_Removed"]].copy()
df_derivative_removed = df_derivative_marked[df_derivative_marked["Derivative_Manual_Removed"]].copy()

print(f"Derivative points before manual removal: {len(df_derivative_original):,}")
print(f"Derivative points manually removed: {len(df_derivative_removed):,}")
print(f"Derivative points final: {len(df_derivative_manual):,}")

## Zusammenfassung

In [ ]:
derivative_summary = (
    df_derivative_marked
    .groupby(["Source_File", "Material", "Temperature", "Mode"], dropna=False)
    .agg(
        derivative_points=("Eps_real_derivative", "size"),
        derivative_removed=("Derivative_Manual_Removed", "sum"),
    )
    .reset_index()
)
manual_summary = derivative_summary.copy()
manual_summary["derivative_removed_percent"] = 100 * manual_summary["derivative_removed"] / manual_summary["derivative_points"]
manual_summary.style.format({"derivative_removed_percent": "{:.2f}"})

## Interaktiver Kontrollplot

In [ ]:
def nearest_available_time(df, target_time, time_col=TIME_COL):
    available_times = np.sort(df[time_col].dropna().unique())
    if len(available_times) == 0:
        raise ValueError("No available times found.")
    return available_times[np.abs(available_times - target_time).argmin()]


def filter_series(df, material=MATERIAL, temperature=TEMPERATURE, mode=MODE):
    return df[(df["Material"] == material) & (df["Temperature"] == temperature) & (df["Mode"] == mode)].copy()


def plot_reference_spectrum(material=MATERIAL, temperature=TEMPERATURE, mode=MODE, target_time=PLOT_TIME_S):
    base = filter_series(df_interpolated_reference, material, temperature, mode)
    if base.empty:
        raise ValueError("No interpolated data found for this selection.")
    plot_time = nearest_available_time(base, target_time)
    plot_df = base[base[TIME_COL] == plot_time].sort_values("Freq_Hz")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(plot_df["Freq_Hz"], plot_df["Eps_real"], color="black", marker="o", markersize=3, linewidth=1, label="reference spectrum")
    ax.set_xscale("log")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Eps real")
    ax.set_title(f"Reference spectrum at t_rel = {plot_time:.2f} s | {material} {temperature} {mode}")
    ax.grid(True, which="both")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return plot_df


def plot_manual_derivative_check(material=MATERIAL, temperature=TEMPERATURE, mode=MODE, target_time=PLOT_TIME_S):
    original_base = filter_series(df_derivative_original, material, temperature, mode)
    marked_base = filter_series(df_derivative_marked, material, temperature, mode)
    if original_base.empty or marked_base.empty:
        raise ValueError("No derivative data found for this selection.")
    plot_time = nearest_available_time(marked_base, target_time)
    original_plot = original_base[original_base[TIME_COL] == nearest_available_time(original_base, plot_time)].sort_values("Omega_rad_s")
    marked_plot = marked_base[marked_base[TIME_COL] == plot_time].sort_values("Omega_rad_s")
    kept = marked_plot[~marked_plot["Derivative_Manual_Removed"]]
    removed = marked_plot[marked_plot["Derivative_Manual_Removed"]]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(original_plot["Omega_rad_s"], original_plot["Eps_real_derivative"], color="0.75", linewidth=1, label="original derivative")
    ax.scatter(kept["Omega_rad_s"], kept["Eps_real_derivative"], color="black", s=24, label="used")
    ax.scatter(removed["Omega_rad_s"], removed["Eps_real_derivative"], color="red", marker="x", s=46, label="removed")
    ax.set_xscale("log")
    if (marked_plot["Eps_real_derivative"] > 0).all() and (original_plot["Eps_real_derivative"] > 0).all():
        ax.set_yscale("log")
    else:
        ax.set_yscale("symlog", linthresh=1)
    ax.set_xlabel("Angular frequency omega (rad/s)")
    ax.set_ylabel("Eps real derivative")
    ax.set_title(f"Manual derivative check at t_rel = {plot_time:.2f} s | {material} {temperature} {mode}")
    ax.grid(True, which="both")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return marked_plot


def plot_clean_derivative_only(material=MATERIAL, temperature=TEMPERATURE, mode=MODE, target_time=PLOT_TIME_S):
    clean_base = filter_series(df_derivative_manual, material, temperature, mode)
    if clean_base.empty:
        raise ValueError("No cleaned derivative data found for this selection.")
    plot_time = nearest_available_time(clean_base, target_time)
    plot_df = clean_base[clean_base[TIME_COL] == plot_time].sort_values("Omega_rad_s")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(plot_df["Omega_rad_s"], plot_df["Eps_real_derivative"], color="black", marker="o", markersize=4, linewidth=1, label="remaining points")
    ax.set_xscale("log")
    if (plot_df["Eps_real_derivative"] > 0).all():
        ax.set_yscale("log")
    else:
        ax.set_yscale("symlog", linthresh=1)
    ax.set_xlabel("Angular frequency omega (rad/s)")
    ax.set_ylabel("Eps real derivative")
    ax.set_title(f"Cleaned derivative only at t_rel = {plot_time:.2f} s | {material} {temperature} {mode}")
    ax.grid(True, which="both")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return plot_df


# Static checks are available by calling these functions manually.
# plot_reference_spectrum()
# plot_manual_derivative_check()
# plot_clean_derivative_only()

In [ ]:
def interactive_manual_check(initial_material=MATERIAL, initial_temperature=TEMPERATURE, initial_mode=MODE, initial_time=PLOT_TIME_S):
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError as exc:
        raise ImportError("Install ipywidgets to use the interactive controls.") from exc

    available = df_interpolated_reference[["Material", "Temperature", "Mode"]].drop_duplicates().sort_values(["Material", "Temperature", "Mode"])
    materials = sorted(available["Material"].dropna().unique())

    material_dropdown = widgets.Dropdown(options=materials, value=initial_material if initial_material in materials else materials[0], description="Material")
    temperature_dropdown = widgets.Dropdown(description="Temp")
    mode_dropdown = widgets.Dropdown(description="Mode")
    view_toggle = widgets.ToggleButtons(options=[("Reference", "reference"), ("Derivative compare", "derivative"), ("Clean only", "clean")], value="clean", description="View")
    time_slider = widgets.SelectionSlider(
        options=[("loading", 0.0)],
        value=0.0,
        description="t_rel",
        continuous_update=False,
        layout=widgets.Layout(width="95%"),
        style={"description_width": "60px"},
    )
    output = widgets.Output()

    def refresh_options(*args):
        mat = material_dropdown.value
        temps = sorted(available.loc[available["Material"] == mat, "Temperature"].dropna().unique())
        temperature_dropdown.options = temps
        if initial_temperature in temps:
            temperature_dropdown.value = initial_temperature
        elif temps:
            temperature_dropdown.value = temps[0]

    def refresh_modes(*args):
        mat = material_dropdown.value
        temp = temperature_dropdown.value
        modes = sorted(available.loc[(available["Material"] == mat) & (available["Temperature"] == temp), "Mode"].dropna().unique())
        mode_dropdown.options = modes
        if initial_mode in modes:
            mode_dropdown.value = initial_mode
        elif modes:
            mode_dropdown.value = modes[0]

    def refresh_times(*args):
        base = filter_series(df_interpolated_reference, material_dropdown.value, temperature_dropdown.value, mode_dropdown.value)
        times = np.sort(base[TIME_COL].dropna().unique())
        if len(times) == 0:
            time_slider.options = [("No data", 0.0)]
            time_slider.value = 0.0
            return
        nearest = times[np.abs(times - initial_time).argmin()]
        time_slider.options = [(f"{time / 60:.2f} min ({time:.1f} s)", float(time)) for time in times]
        time_slider.value = float(nearest)

    def redraw(*args):
        with output:
            output.clear_output(wait=True)
            if not time_slider.options:
                print("No data for this selection.")
                return
            if view_toggle.value == "reference":
                display(plot_reference_spectrum(material_dropdown.value, temperature_dropdown.value, mode_dropdown.value, time_slider.value))
            elif view_toggle.value == "derivative":
                display(plot_manual_derivative_check(material_dropdown.value, temperature_dropdown.value, mode_dropdown.value, time_slider.value))
            else:
                display(plot_clean_derivative_only(material_dropdown.value, temperature_dropdown.value, mode_dropdown.value, time_slider.value))

    material_dropdown.observe(refresh_options, names="value")
    temperature_dropdown.observe(refresh_modes, names="value")
    mode_dropdown.observe(refresh_times, names="value")
    view_toggle.observe(redraw, names="value")
    clear_output(wait=True)
    time_slider.observe(redraw, names="value")

    refresh_options()
    refresh_modes()
    refresh_times()
    controls = widgets.VBox([widgets.HBox([material_dropdown, temperature_dropdown, mode_dropdown, view_toggle]), time_slider, output])
    display(controls)
    redraw()


# Optional full control panel. The clean-only slider is in the next cell.
# interactive_manual_check()

## Zeitschieber nur fuer bereinigte Ableitung

In [ ]:
def interactive_clean_derivative_spectrum(
    df,
    material=MATERIAL,
    temperature=TEMPERATURE,
    mode=MODE,
    initial_time=PLOT_TIME_S,
):
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError as exc:
        raise ImportError("Install ipywidgets to use the interactive derivative slider.") from exc

    plot_base = filter_series(df, material, temperature, mode)
    if plot_base.empty:
        raise ValueError("No cleaned derivative data found for this selection.")

    available_times = np.sort(plot_base[TIME_COL].dropna().unique())
    initial_available_time = nearest_available_time(plot_base, initial_time)
    slider_options = [
        (f"{time / 60:.2f} min ({time:.1f} s)", float(time))
        for time in available_times
    ]

    time_slider = widgets.SelectionSlider(
        options=slider_options,
        value=float(initial_available_time),
        description="t_rel",
        continuous_update=False,
        layout=widgets.Layout(width="95%"),
        style={"description_width": "60px"},
    )

    output = widgets.Output()
    controls = widgets.VBox([time_slider, output])

    def redraw(change=None):
        with output:
            output.clear_output(wait=True)
            display(plot_clean_derivative_only(
                material=material,
                temperature=temperature,
                mode=mode,
                target_time=time_slider.value,
            ))

    time_slider.observe(redraw, names="value")
    display(controls)
    redraw()


interactive_clean_derivative_spectrum(
    df_derivative_manual,
    material=MATERIAL,
    temperature=TEMPERATURE,
    mode=MODE,
    initial_time=PLOT_TIME_S,
)

## Manipulierte Daten separat speichern und wieder laden

In [ ]:
def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def export_manual_manipulation(export_dir=EXPORT_DIR):
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    written_files = []
    source_files = sorted(df_derivative_manual["Source_File"].dropna().unique())
    for source_file in source_files:
        stem = Path(str(source_file)).stem
        source_manual = df_derivative_manual[df_derivative_manual["Source_File"] == source_file]
        source_removed = df_derivative_removed[df_derivative_removed["Source_File"] == source_file]
        source_summary = manual_summary[manual_summary["Source_File"] == source_file]

        output_paths = [
            export_dir / f"{stem}_manipulated.csv",
            export_dir / f"{stem}_removed_derivative_points.csv",
            export_dir / f"{stem}_manual_summary.csv",
            export_dir / f"{stem}_manual_rules.json",
        ]

        source_manual.to_csv(output_paths[0], index=False)
        source_removed.to_csv(output_paths[1], index=False)
        source_summary.to_csv(output_paths[2], index=False)
        write_json(output_paths[3], {
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "manipulation_name": MANIPULATION_NAME,
            "selected_file": source_file,
            "material_filter": MATERIAL_FILTER,
            "manual_derivative_removals": MANUAL_DERIVATIVE_REMOVALS,
            "written_files": [str(path) for path in output_paths[:3]],
        })
        written_files.extend(str(path) for path in output_paths)

    return written_files


def load_manual_manipulation(export_dir=EXPORT_DIR):
    export_dir = Path(export_dir)
    return {
        "eps_real_derivative_manual": pd.concat(
            [pd.read_csv(path) for path in sorted(export_dir.glob("*_manipulated.csv"))],
            ignore_index=True,
        ),
        "eps_real_derivative_removed_points": pd.concat(
            [pd.read_csv(path) for path in sorted(export_dir.glob("*_removed_derivative_points.csv"))],
            ignore_index=True,
        ),
        "manual_summary": pd.concat(
            [pd.read_csv(path) for path in sorted(export_dir.glob("*_manual_summary.csv"))],
            ignore_index=True,
        ),
        "manual_rules": [
            json.loads(path.read_text(encoding="utf-8"))
            for path in sorted(export_dir.glob("*_manual_rules.json"))
        ],
    }


SAVE_MANIPULATED_DATA = False

if SAVE_MANIPULATED_DATA:
    written_files = export_manual_manipulation()
    print("Manipulated data written:")
    for path in written_files:
        print(path)
else:
    print("Export disabled. Set SAVE_MANIPULATED_DATA = True in this cell when the inspected data look good.")